# NanoNemo on Kaggle 2x T4

Thin launcher: everything runs from the repo's scripts. Settings: **Accelerator: GPU T4 x2**, Internet on.
Inputs: the token dataset (`nanonemo-nemotron-cc`), and for a resumed main run, the previous session's output.
See `docs/kaggle_workflow.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/SabariIyyappan/NanoNemo.git
%cd NanoNemo
!pip install -q -r requirements-kaggle.txt && pip install -q --no-deps -e .
!python scripts/smoke_gpu.py | tail -5

In [ ]:
import glob, os
manifests = [m for m in glob.glob('/kaggle/input/**/manifest.json', recursive=True)]
DATA = os.path.dirname(manifests[0])
CKPTS = sorted(glob.glob('/kaggle/input/**/ckpt/latest', recursive=True))
PREV = os.path.dirname(CKPTS[-1]) if CKPTS else None
print('data:', DATA, '| previous checkpoint:', PREV)
os.environ['DATA'] = DATA

## M7: 1-GPU vs 2-GPU parity and resume tests (real GPUs, NCCL)

In [ ]:
!NANONEMO_TEST_DEVICE=cuda python -m pytest -q -m multigpu tests/distributed

## M7: multi-GPU throughput table (+ an nsys trace per row for the communication share)

In [ ]:
!python scripts/bench_train.py --config configs/dist/single.yaml --steps 10 --data-dir $DATA --out results/dist/t4-x1/single.json
!for c in ddp ddp_tuned fsdp fsdp_prefetch fsdp_compile_blocks tp; do \
    torchrun --standalone --nproc_per_node=2 scripts/bench_train.py --config configs/dist/$c.yaml --steps 10 --data-dir $DATA; done
!python scripts/dist_table.py

In [ ]:
!for c in ddp ddp_tuned fsdp_prefetch tp; do scripts/nsys_dist.sh configs/dist/$c.yaml 2 --data-dir $DATA; done

## M7: LR sweep (3 short runs, ~35 min each)

In [ ]:
!for lr in 3.0e-4 6.0e-4 1.2e-3; do \
    torchrun --standalone --nproc_per_node=2 scripts/train.py --config configs/lr_sweep/lr_$lr.yaml \
        --set train.data_dir=$DATA train.out_dir=/kaggle/working/lr_sweep/lr_$lr | grep -E 'val_loss|step +[0-9]*0 '; done

## M9: main run (resumes automatically if a previous session's output is attached)

In [ ]:
resume = f'train.resume_from={PREV}' if PREV else ''
!torchrun --standalone --nproc_per_node=2 scripts/train.py --config configs/main.yaml \
    --set train.data_dir=$DATA train.out_dir=/kaggle/working/main train.stop_after_min=690 {resume}

## Save results (Output panel; download, unzip at the repo root)

In [ ]:
!zip -qr /kaggle/working/results.zip results
!ls -la /kaggle/working